In [16]:
import pandas as pd
from typing import List, Dict, Tuple

def p_star_teorico(
    ruta_ciudades: str,
    ruta_hospitales: str,
    H: int = 30,
    LOS: int = 5,
    occ_target: float = 0.85,   # holgura/ocupación objetivo (<=1). Baja este valor para más colchón.
    rhos: List[float] = [0.005, 0.01, 0.02, 0.05, 0.08],  # 0.5%, 1%, 2%, 5%, 8%
    p_grid: List[int] = [10,12,15,18,20,25,30,35,40]
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Devuelve:
      - df_grid: tabla larga con filas (rho, p) y métricas (Q, C_top_p, util=Q/C_top_p, feasible)
      - df_pstar: resumen por rho con p* mínimo factible y métricas asociadas
    """
    ciudades = pd.read_csv(ruta_ciudades)      # requiere columna 'poblacion'
    hospitales = pd.read_csv(ruta_hospitales)  # requiere columna 'capacidad'

    # Capacidad rotatoria en H días con ocupación objetivo (holgura)
    hospitales = hospitales.copy()
    hospitales["C"] = hospitales["capacidad"] * (H / LOS) * occ_target
    hospitales = hospitales.sort_values("C", ascending=False).reset_index(drop=True)

    # Precalcula acumulado de capacidad top-p
    hospitales["C_cum"] = hospitales["C"].cumsum()

    rows = []
    pstar_rows = []

    for rho in rhos:
        # Demanda total esperada en H días
        Q = float(ciudades["poblacion"].sum() * rho)

        # Para cada p del grid, mira si la capacidad acumulada top-p cubre Q
        p_found = None
        for p in p_grid:
            if p > len(hospitales):
                # por si el grid sobrepasa nº hospitales disponibles
                C_top_p = hospitales["C"].sum()
            else:
                C_top_p = float(hospitales.loc[p-1, "C_cum"])  # acumulado hasta índice p-1

            util = Q / C_top_p if C_top_p > 0 else float("inf")
            feasible = (C_top_p >= Q)

            rows.append({
                "rho": rho,
                "p": p,
                "Q_demand": Q,
                "C_top_p": C_top_p,
                "utilization_Q_over_C": util,  # <1 implica holgura
                "feasible": int(feasible)      # 1 si cubre Q con la holgura deseada
            })

            if feasible and p_found is None:
                p_found = (p, Q, C_top_p, util)

        # Si existe p* para este rho, guárdalo; si no, marca sin factibilidad
        if p_found is not None:
            p_opt, Qopt, Copt, utilopt = p_found
            pstar_rows.append({
                "rho": rho,
                "p_star": p_opt,
                "Q_demand": Qopt,
                "C_top_p_star": Copt,
                "utilization_at_p_star": utilopt,  # cuanto menor, más colchón
                "coverage_margin": (Copt - Qopt)   # margen absoluto de capacidad
            })
        else:
            pstar_rows.append({
                "rho": rho,
                "p_star": None,
                "Q_demand": Q,
                "C_top_p_star": hospitales["C"].sum(),
                "utilization_at_p_star": Q / max(hospitales["C"].sum(), 1e-9),
                "coverage_margin": hospitales["C"].sum() - Q
            })

    df_grid = pd.DataFrame(rows)
    df_pstar = pd.DataFrame(pstar_rows)

    # Orden bonito
    df_grid = df_grid.sort_values(["rho", "p"]).reset_index(drop=True)
    df_pstar = df_pstar.sort_values("rho").reset_index(drop=True)
    return df_grid, df_pstar

# === Ejemplo de uso ===
df_grid, df_pstar = p_star_teorico(
    ruta_ciudades="../data/processed/andaluces_2_5k.csv",
    ruta_hospitales="../data/processed/Hospitales_Completo.csv",
    H=30, LOS=5, occ_target=1,  # occ_target más bajo = más holgura
    rhos=[0.0001, 0.0005,0.001,0.005, 0.01],
    p_grid=[10,15,20,30,50]
)
# df_grid.to_csv("runs/teorico_grid.csv", index=False)
# df_pstar.to_csv("runs/teorico_pstar.csv", index=False)
display(df_pstar)


,rho,p_star,Q_demand,C_top_p_star,utilization_at_p_star,coverage_margin
0,0.0001,10,818.5809,41172.0,0.019882,40353.4191
1,0.0005,10,4092.9045,41172.0,0.099410,37079.0955
2,0.0010,10,8185.8090,41172.0,0.198820,32986.1910
3,0.0050,10,40929.0450,41172.0,0.994099,242.9550
4,0.0100,50,81858.0900,105012.0,0.779512,23153.9100


In [37]:
from typing import List, Tuple, Optional
import numpy as np
import matplotlib.pyplot as plt
def p_star_teorico(
    ruta_ciudades: str,
    ruta_hospitales: str,
    H: int = 30,
    LOS: int = 5,
    occ_target: float = 0.85,
    rhos: List[float] = [0.005, 0.01, 0.02, 0.05, 0.08],   # 0.5%, 1%, 2%, 5%, 8%
    p_grid: List[int] = [10,12,15,18,20,25,30,35,40,45,50,60,80]
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Devuelve:
      - df_grid: filas (rho, p) con Q, C_top_p, utilización y factibilidad teórica
      - df_pstar: por rho, el mínimo p que cubre Q (teórico, sin geografía)
    Además guarda CSV en runs/.
    """

    ciudades = pd.read_csv(ruta_ciudades)      # requiere 'poblacion'
    hospitales = pd.read_csv(ruta_hospitales)  # requiere 'capacidad'

    # Capacidad rotatoria en H días con ocupación objetivo (holgura)
    hos = hospitales.copy()
    hos["C"] = hos["capacidad"] * (H / LOS) * occ_target
    hos = hos.sort_values("C", ascending=False).reset_index(drop=True)
    hos["C_cum"] = hos["C"].cumsum()

    rows = []
    pstar_rows = []

    for rho in rhos:
        Q = float(ciudades["poblacion"].sum() * rho)
        p_found = None

        for p in p_grid:
            # Acumulado top-p
            if p > len(hos):
                C_top_p = float(hos["C"].sum())
            else:
                C_top_p = float(hos.loc[p-1, "C_cum"])
            util = Q / C_top_p if C_top_p > 0 else float("inf")
            feasible = int(C_top_p >= Q)

            rows.append({
                "rho": rho,
                "p": p,
                "Q_demand": Q,
                "C_top_p": C_top_p,
                "utilization_Q_over_C": util,
                "feasible": feasible
            })
            if feasible and p_found is None:
                p_found = (p, Q, C_top_p, util)

        if p_found is not None:
            p_opt, Qopt, Copt, utilopt = p_found
            pstar_rows.append({
                "rho": rho,
                "p_star": p_opt,
                "Q_demand": Qopt,
                "C_top_p_star": Copt,
                "utilization_at_p_star": utilopt,
                "coverage_margin": (Copt - Qopt)
            })
        else:
            # No cubre ni con todos
            C_all = float(hos["C"].sum())
            pstar_rows.append({
                "rho": rho,
                "p_star": None,
                "Q_demand": Q,
                "C_top_p_star": C_all,
                "utilization_at_p_star": (Q / max(C_all, 1e-9)),
                "coverage_margin": (C_all - Q)
            })

    df_grid = pd.DataFrame(rows).sort_values(["rho", "p"]).reset_index(drop=True)
    df_pstar = pd.DataFrame(pstar_rows).sort_values("rho").reset_index(drop=True)

    return df_grid, df_pstar


# -------------------------------------------------
# 2) Curva C_top_p vs p (no depende de rho)
# -------------------------------------------------
def plot_capacity_curve(
    ruta_hospitales: str,
    H: int = 30,
    LOS: int = 5,
    occ_target: float = 0.85,
    p_max: Optional[int] = None
) -> None:
    """
    Traza C_top_p (capacidad acumulada) vs p con ocupación objetivo occ_target.
    """
   

    hospitales = pd.read_csv(ruta_hospitales)
    hos = hospitales.copy()
    hos["C"] = hos["capacidad"] * (H / LOS) * occ_target
    hos = hos.sort_values("C", ascending=False).reset_index(drop=True)
    hos["C_cum"] = hos["C"].cumsum()

    n = len(hos) if p_max is None else min(p_max, len(hos))
    p_vals = np.arange(1, n + 1)
    C_vals = hos.loc[:n-1, "C_cum"].values

    plt.figure()
    plt.plot(p_vals, C_vals)
    plt.xlabel("p (nº de hospitales)")
    plt.ylabel("Capacidad acumulada en H días (personas)")
    plt.title("Curva C_top_p vs p (capacidad acumulada con holgura)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# === Ejemplo de uso ===
DIST = "../data/matrix/time_ciudad_hospital_min.csv"
CITIES = "../data/processed/Ciudades_Con_Demanda.csv"
HOSPS = "../data/processed/Hospitales_Con_Capacidad.csv"

# Parámetros globales
H = 30
LOS = 5
OCC = 0.85
RHOS = [0.001, 0.005, 0.01]
P_GRID_EXT = [5,8,10,15,20,25,30,60,70]

# 1) Tablas teóricas con p_grid extendido
df_grid, df_pstar = p_star_teorico(
    ruta_ciudades=CITIES, ruta_hospitales=HOSPS,
    H=H, LOS=LOS, occ_target=OCC,
    rhos=RHOS, p_grid=P_GRID_EXT
)
display(df_pstar)
print(df_grid[df_grid['feasible']==1])


,rho,p_star,Q_demand,C_top_p_star,utilization_at_p_star,coverage_margin
0,0.001,5,8185.809,19206.6,0.426198,11020.791
1,0.005,15,40929.045,47011.8,0.870612,6082.755
2,0.010,60,81858.090,96420.6,0.848969,14562.510


      rho   p   Q_demand   C_top_p  utilization_Q_over_C  feasible
0   0.001   5   8185.809   19206.6              0.426198         1
1   0.001   8   8185.809   28906.8              0.283179         1
2   0.001  10   8185.809   34996.2              0.233906         1
3   0.001  15   8185.809   47011.8              0.174122         1
4   0.001  20   8185.809   55590.0              0.147253         1
5   0.001  25   8185.809   62837.1              0.130270         1
6   0.001  30   8185.809   69089.7              0.118481         1
7   0.001  60   8185.809   96420.6              0.084897         1
8   0.001  70   8185.809  102081.6              0.080189         1
12  0.005  15  40929.045   47011.8              0.870612         1
13  0.005  20  40929.045   55590.0              0.736266         1
14  0.005  25  40929.045   62837.1              0.651352         1
15  0.005  30  40929.045   69089.7              0.592404         1
16  0.005  60  40929.045   96420.6              0.424484      